Remember to launch this from a directory where the environment has been correctly configured

In [ ]:
import omnidsp_py
import numpy as np
import librosa
from matplotlib import pyplot as plt

In [ ]:
# Example using rfft (double precision)
N = 8192
freq = 20.0
signal = np.cos(2.0 * np.pi * freq * np.arange(N) / N).astype(np.float64)

# Use the bound convenience function
spectrum = omnidsp_py.rfft(signal)

print(f"Input signal shape: {signal.shape}")
print(f"Output spectrum shape: {spectrum.shape}")  # Should be N/2 + 1
print("Spectrum from omnidsp_py (first 5 bins):")
print(spectrum[:5])

np_spec = np.fft.rfft(signal)
print("Spectrum from numpy (first 5 bins):")
print(np_spec[:5])

In [ ]:
sample_rate = 44100.0
fmin = 15.0
fmax = 1000.0
bins_per_octave = 12


def hann_generator(dummy_arr):
    """Generates a Hann window with the same size and dtype as a dummy array.

    This function takes a dummy array (e.g., potentially passed from C++),
    determines its required length and data type, creates a placeholder array
    of ones with those characteristics, applies the OmniDSP Hann window
    function to the ones array, and returns the resulting window coefficients.

    Args:
        dummy_arr (np.ndarray): An array whose length and dtype are used to
            determine the output window's properties. The values are ignored.

    Returns:
        np.ndarray: The generated Hann window coefficients with the same length
            and dtype as dummy_arr.
    """
    return omnidsp_py.Window.hann(np.ones(len(dummy_arr), dtype=dummy_arr.dtype))


# --- Use the factory function ---
try:
    cqt_plan = omnidsp_py.create_cqt_plan(
        sample_rate=sample_rate,
        lowest_freq=fmin,
        highest_freq=fmax,
        bins_per_octave=bins_per_octave,
        window_function=hann_generator,  # Pass the Python lambda
        precision=omnidsp_py.Precision.DOUBLE,  # Specify precision
    )

    # Execute the plan (assuming 'signal' is defined and is float64)
    if "signal" in locals() and signal.dtype == np.float64:
        omnidsp_cqt_complex = cqt_plan.execute(signal)
        omnidsp_cqt_amp = np.abs(omnidsp_cqt_complex)
        print(f"CQT Output shape: {omnidsp_cqt_amp.shape}")
        # print(f"First 5 CQT amplitudes: {omnidsp_cqt_amp[:5]}")
    else:
        print("Please define 'signal' as a float64 NumPy array before executing.")

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
librosa_cqt_amp = np.abs(
    librosa.cqt(
        signal,
        sr=sample_rate,
        n_bins=72,
        fmin=fmin,
        window="hann",
    )
).flatten()

print(f"CQT Output shape: {librosa_cqt_amp.shape}")

In [ ]:
plt.plot(omnidsp_cqt_amp)

In [ ]:
plt.plot(librosa_cqt_amp)